# dfs-three-set-toposort — ex3: deterministic toposort — sort children by key for reproducible output

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `dfs-three-set-toposort`. Running the final beacon cell reports progress against the `Backprop: DFS three-set toposort` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: DFS three-set toposort` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`dfs-three-set-toposort`** (exercise 3). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "dfs-three-set-toposort"
DD_SUBTOPIC = "Backprop: DFS three-set toposort"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Deterministic toposort — sort children by key for reproducible output

Ex1's recursive toposort and ex2's iterative variant both produce a
valid order — but a different valid order if `get_children` returns
children in different iteration orders. The deepening move enforces
DETERMINISM by sorting children by a caller-supplied key before
recursing.

```python
def topological_sort_keyed(root, get_children, key):
    perm, temp = set(), set()
    result = []
    def visit(node):
        nid = id(node)
        if nid in perm: return
        if nid in temp: raise ValueError('cycle')
        temp.add(nid)
        # Sort children BEFORE recursing → output is now key-deterministic.
        for child in sorted(get_children(node), key=key):
            visit(child)
        temp.remove(nid)
        perm.add(nid)
        result.append(node)
    visit(root)
    return result
```

**Why determinism matters.** Two CI runs over the same graph should
yield the same toposort. Without a tiebreaker, dict-iteration order
(Python 3.7+ preserves insertion order but the INSERTION order can
still vary by user code) leaks into the result, making cache
invalidation and test diffs noisy.

**Deps-first invariant is unchanged.** Sorting children only reorders
SIBLINGS at each level. Every node still appears AFTER all of its
(transitive) children — the toposort guarantee survives the
tiebreaker.

**`key=key`, not `key=str`.** Hardcoding `str` would force nodes to
have a string representation; the caller-passed key gives the user
control (e.g. by node name, by topo-index, by registration order).

### Exercise 3 — deterministic toposort — sort children by key for reproducible output

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the three-color recursive toposort with a caller-supplied `key` callable that orders children at every level — producing an output that is deterministic under arbitrary `get_children` iteration order while preserving the deps-first invariant.
> Keywords: toposort, deterministic, tiebreaker, sort-key, reproducibility
> ```

**KCs targeted:** `dfs-three-set-toposort`, `child-key-tiebreaker`

Implement `ex3_topological_sort_keyed(root, get_children, key)`. Same contract as ex1's recursive `topological_sort`, but with a tiebreaker:

- `root`: the start node.
- `get_children(node) -> list` returning the node's direct successors.
- `key`: a callable `node -> comparable`. Children are sorted by this key BEFORE recursing.

Output: `list[node]` in deps-FIRST order (root LAST). Same one-appearance-per-node and cycle invariants as ex1:
- `ValueError('cycle')` on a back-edge.
- Each reachable node appears EXACTLY once (diamond DAGs OK).
- `root` is the LAST element of the returned list.

**Determinism is the headline.** Calling the function twice on the same graph with the same `key` MUST return identical output, regardless of `get_children` iteration order.

Use `id(node)` for set keys (temp + perm sets, as in ex1/ex2). Children sort uses the caller-supplied `key`, NOT `id`.

Algorithm sketch:

```python
def ex3_topological_sort_keyed(root, get_children, key):
    perm, temp = set(), set()
    result = []
    def visit(node):
        nid = id(node)
        if nid in perm: return
        if nid in temp: raise ValueError('cycle')
        temp.add(nid)
        for child in sorted(get_children(node), key=key):
            visit(child)
        temp.remove(nid); perm.add(nid)
        result.append(node)
    visit(root)
    return result
```

In [ ]:
def ex3_topological_sort_keyed(root, get_children, key) -> list:
    """Deterministic three-color toposort; children sorted by `key` at each level."""
    raise NotImplementedError()


def _test_ex3():
    def _test_ex3():
        class N:
            def __init__(self, name):
                self.name = name
                self.children = []
            def __repr__(self):
                return f'N({self.name})'
        def kids(n):
            return n.children
        def by_name(n):
            return n.name

        # === Linear chain: a -> b -> c.  deps-first → [c, b, a] ===
        a, b, c = N('a'), N('b'), N('c')
        a.children = [b]
        b.children = [c]
        out = ex3_topological_sort_keyed(a, kids, by_name)
        assert [n.name for n in out] == ['c', 'b', 'a'], [n.name for n in out]
        assert out[-1] is a, 'root must be LAST'

        # === Determinism: child order in get_children should NOT affect output. ===
        a, b, c, d = N('a'), N('b'), N('c'), N('d')
        a.children = [b, c]
        b.children = [d]
        c.children = [d]
        out1 = ex3_topological_sort_keyed(a, kids, by_name)
        # Reverse child order in a — should still yield the same sorted output.
        a.children = [c, b]
        out2 = ex3_topological_sort_keyed(a, kids, by_name)
        assert [n.name for n in out1] == [n.name for n in out2], (
            f'output not deterministic across child orderings: {[n.name for n in out1]} vs {[n.name for n in out2]}'
        )

        # === Diamond DAG: each node appears EXACTLY once ===
        assert len(out1) == 4 and len({id(n) for n in out1}) == 4
        # And root is last:
        assert out1[-1] is a
        # And d appears before b and c (deps-first):
        names = [n.name for n in out1]
        assert names.index('d') < names.index('b'), names
        assert names.index('d') < names.index('c'), names
        # Sibling ordering: among b vs c, alphabetical (key=by_name) → b before c.
        assert names.index('b') < names.index('c'), f'sibling order should be alphabetical: {names}'

        # === Cycle raises ValueError ===
        a, b = N('a'), N('b')
        a.children = [b]
        b.children = [a]
        try:
            ex3_topological_sort_keyed(a, kids, by_name)
            assert False, 'cycle should raise'
        except ValueError:
            pass

        # === Numeric key — non-string sort works ===
        n1, n2, n3 = N('a'), N('b'), N('c')
        n1.children = [n2, n3]
        n2.priority = 5
        n3.priority = 1
        n1.priority = 0
        by_priority = lambda nd: nd.priority
        out = ex3_topological_sort_keyed(n1, kids, by_priority)
        # Children sorted by priority: n3 (1) then n2 (5).  So order [n3, n2, n1].
        assert [n.name for n in out] == ['c', 'b', 'a'], [n.name for n in out]

        # === Caller can override default ordering. ===
        # Same graph as the diamond, but key=lambda n: -ord(n.name[0]) reverses sibling order.
        a, b, c, d = N('a'), N('b'), N('c'), N('d')
        a.children = [b, c]
        b.children = [d]
        c.children = [d]
        reverse_alpha = lambda n: -ord(n.name[0])
        out = ex3_topological_sort_keyed(a, kids, reverse_alpha)
        names = [n.name for n in out]
        # Now sibling order is c before b (reversed):
        assert names.index('c') < names.index('b'), names
        print('ex3 ok')

    _test_ex3()
    _dd_passed.add('ex3')
    print("ex3 ✓")

_test_ex3()

<details><summary>Solution</summary>

```python
def ex3_topological_sort_keyed(root, get_children, key):
    perm, temp = set(), set()
    result = []

    def visit(node):
        nid = id(node)
        if nid in perm:
            return
        if nid in temp:
            raise ValueError('cycle')
        temp.add(nid)
        # Sort children BEFORE recursing — deterministic sibling order.
        for child in sorted(get_children(node), key=key):
            visit(child)
        temp.remove(nid)
        perm.add(nid)
        result.append(node)

    visit(root)
    return result
```

**Sort children, not the result.** Sorting the result list would destroy the toposort invariant (children must come before parents). Sorting siblings AT EACH LEVEL only reorders nodes that are topologically incomparable — the deps-first guarantee survives.

**`sorted(..., key=key)` is stable.** Python's sort is stable, so ties in `key(node)` preserve the order from `get_children`. The test uses unique keys to avoid this corner.

**`id(node)` for sets, `key(node)` for the order.** Identity vs ordering are different concerns. Sets dedup by identity (so diamond DAGs don't double-visit); the `key` callable only determines sibling traversal order.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex3',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()